# Recunoaștere locală vs. globală

**Câte ediții Wikipedia vorbesc despre fiecare persoană onorată?**

Numărul de ediții lingvistice ale Wikipedia care au un articol despre o persoană este un proxy decent pentru notorietatea internațională. Datele vin din Wikidata (câmpul `sitelinks`).

Praguri: Universal ≥50, Național 5–49, Local 1–4.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({'font.family':'serif','figure.dpi':130,
                     'axes.spines.top':False,'axes.spines.right':False})
ACCENT, INK, MUTED = '#C04F35', '#15171A', '#6E6E70'

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row
print('Connected.')

## 1. Distribuția sitelink-urilor

In [ ]:
df = pd.read_sql("""
    SELECT p.full_name, p.wiki_sitelinks, p.wiki_scope, p.wiki_ro_views,
           p.profession, p.era, COUNT(*) AS streets
    FROM streets_dedup sd
    JOIN persons p ON p.core_name_norm = sd.core_name_norm
    WHERE p.wiki_sitelinks IS NOT NULL
    GROUP BY p.core_name_norm
""", conn)

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(df['wiki_sitelinks'], bins=40, color=INK, alpha=0.7, edgecolor='white')
ax.axvline(50, color=ACCENT, linestyle='--', label='Prag universal (50)')
ax.axvline(5,  color=MUTED,  linestyle='--', label='Prag național (5)')
ax.set_xlabel('Număr de ediții Wikipedia')
ax.set_ylabel('Persoane')
ax.set_title('Distribuția notorietății internaționale (sitelinks Wikidata)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(df['wiki_scope'].value_counts().to_string())

## 2. Recunoaștere pe profesie

In [ ]:
df_prof = pd.read_sql("""
    SELECT p.profession, p.wiki_scope, COUNT(DISTINCT p.core_name_norm) AS n
    FROM persons p
    WHERE p.wiki_scope IS NOT NULL AND p.profession IS NOT NULL
    GROUP BY p.profession, p.wiki_scope
""", conn)

pivot = df_prof.pivot_table(index='profession', columns='wiki_scope', values='n', fill_value=0)
for col in ['universal','national','local']:
    if col not in pivot:
        pivot[col] = 0
pivot = pivot[['universal','national','local']]
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False).head(12).drop(columns='total')

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', stacked=True, ax=ax,
           color=[ACCENT, INK, MUTED], width=0.7)
ax.set_title('Recunoaștere internațională pe profesie', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Persoane')
ax.legend(title='Nivel', labels=['Universal','Național','Local'])
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 3. Wikipedia pageviews vs. număr de străzi

In [ ]:
df_pv = df[df['wiki_ro_views'] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 6))
scatter_colors = df_pv['wiki_scope'].map({'universal':ACCENT,'national':INK,'local':MUTED})
ax.scatter(df_pv['streets'], df_pv['wiki_ro_views'], c=scatter_colors, alpha=0.7, s=40)

# Label outliers
threshold_pv = df_pv['wiki_ro_views'].quantile(0.92)
threshold_st = df_pv['streets'].quantile(0.90)
for _, row in df_pv[(df_pv['wiki_ro_views'] > threshold_pv) | (df_pv['streets'] > threshold_st)].iterrows():
    ax.annotate(row['full_name'], (row['streets'], row['wiki_ro_views']),
                fontsize=8, color=MUTED, xytext=(4,4), textcoords='offset points')

from matplotlib.lines import Line2D
legend = [Line2D([0],[0],marker='o',color='w',markerfacecolor=c,markersize=8,label=l)
          for c,l in [(ACCENT,'Universal'),(INK,'Național'),(MUTED,'Local')]]
ax.legend(handles=legend)
ax.set_xlabel('Număr de străzi')
ax.set_ylabel('Vizualizări lunare medii (Wikipedia RO)')
ax.set_title('Popularitate online vs. prezență pe stradă', fontsize=13)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 4. Persoane cu multe străzi dar fără articol Wikipedia

In [ ]:
df_no_wiki = pd.read_sql("""
    SELECT p.full_name, p.profession, p.era, COUNT(*) AS streets
    FROM streets_dedup sd
    JOIN persons p ON p.core_name_norm = sd.core_name_norm
    WHERE (p.wiki_sitelinks IS NULL OR p.wiki_sitelinks = 0)
      AND p.wikidata_qid IS NOT NULL
    GROUP BY p.core_name_norm
    ORDER BY streets DESC
    LIMIT 15
""", conn)

if not df_no_wiki.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(df_no_wiki['full_name'][::-1], df_no_wiki['streets'][::-1], color=MUTED)
    ax.set_title('Persoane cu QID Wikidata dar fără articol Wikipedia (0 sitelinks)', fontsize=12)
    ax.set_xlabel('Număr de străzi')
    plt.tight_layout()
    plt.show()
else:
    print('Toți cu QID au cel puțin un articol.')

In [ ]:
conn.close()